# Notebook 04c: Advanced Kinematic Boosting & Multi-Inductive-Bias Ensembling (WLASL-100)

**Context & Purpose:**
In Notebook 04b, switching from frozen full-frame ResNet50 features to 53-keypoint whole-body pose + left/right hands (349-D) produced a decisive breakthrough:
- **Validation Top-1 jumped from 11.01% -> 41.28%** (~3.7x boost).
- **Validation Top-5 jumped from 25.69% -> 68.81%**.
- **Overfitting was eliminated:** Validation loss dropped steadily from ~4.65 down to **3.080** without the U-turn rebound seen in Notebook 03/04.
- Adding ResNet50 back into the pose model (`mm_pose_resnet_language`) dropped Top-1 to 23.85%, confirming that ImageNet features injected signer-background shortcuts on unseen signers.

**What Notebook 04c Does (Fast 2–3 Minute GPU / 4–5 Minute CPU Training):**
Because all pose features (`P_train.npy`, `P_val.npy`, `P_test.npy`) and text embeddings (`text_embeddings_prompt.pt`) are already extracted and cached on Google Drive, **zero video decoding is needed**. This notebook focuses entirely on model optimization and ensembling:
1. **Enhanced Kinematic Augmentation:** Temporal elastic resampling, coordinate scaling, hand-specific relative jitter, joint confidence dropout, and dynamically recomputed velocity.
2. **Cosine Classifier (Normalized Angular Head):** Replaces unconstrained linear heads with cosine similarity heads (`scale * cos(x, W)`), preventing frequent classes from dominating weight norms and boosting fine-grained class separation.
3. **Diverse Sequential Backbones:**
   - `pose_transformer_cosine`: 2-layer Temporal Transformer + Cosine Classifier.
   - `pose_bilstm_attn`: 2-layer Bidirectional LSTM + Attention Pooling + Cosine Classifier (an entirely different sequential inductive bias).
   - `mm_pose_language_boosted`: Multimodal Transformer aligned with all-MiniLM-L6-v2 prompted text embeddings + Cosine Classifier (100 epochs, cosine schedule with 5-epoch warmup).
4. **Multi-Inductive-Bias Inference Ensemble:** Combines the predictions of the diverse models on the unseen validation split to maximize generalization before the locked test evaluation (Notebook 05).


## 1. Environment Setup & Google Drive Mount


In [ ]:
import json
import math
import os
import random
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    print("Google Drive mounted at /content/drive.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | Device: {DEVICE}")


## 2. Path Configuration & Feature Loading (Zero Video Re-Extraction)


In [ ]:
import json

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100")
else:
    DRIVE_ROOT = Path("./voxel51_wlasl100")

META_DIR = DRIVE_ROOT / "metadata"
FEAT_DIR = DRIVE_ROOT / "features"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
EXP_DIR = DRIVE_ROOT / "experiments"
PLOTS_DIR = DRIVE_ROOT / "reports" / "plots"

for d in (CKPT_DIR, EXP_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Load precomputed 349-D pose arrays from Notebook 04b
P_train = np.load(FEAT_DIR / "P_train.npy").astype(np.float32)
P_val = np.load(FEAT_DIR / "P_val.npy").astype(np.float32)
y_train = np.load(FEAT_DIR / "y_train.npy").astype(np.int64)
y_val = np.load(FEAT_DIR / "y_val.npy").astype(np.int64)

# Load prompted text embeddings from all-MiniLM-L6-v2 (100, 384)
prompt_pt = torch.load(FEAT_DIR / "text_embeddings_prompt.pt", map_location="cpu", weights_only=False)
vocab_prompt = prompt_pt["text_embeddings"].float().to(DEVICE)

with open(META_DIR / "class_to_id.json") as f:
    class_to_id = json.load(f)
with open(META_DIR / "id_to_class.json") as f:
    id_to_class = {int(k): v for k, v in json.load(f).items()}

NUM_CLASSES = len(class_to_id)
POSE_DIM = P_train.shape[-1]
NUM_FRAMES = P_train.shape[1]

print(f"Features ready from Drive:")
print(f"  P_train     : shape={P_train.shape}, dtype={P_train.dtype}, finite={np.isfinite(P_train).all()}")
print(f"  P_val       : shape={P_val.shape}, dtype={P_val.dtype}, finite={np.isfinite(P_val).all()}")
print(f"  vocab_prompt: shape={vocab_prompt.shape}, device={vocab_prompt.device}")
print(f"  Classes     : {NUM_CLASSES} classes | Train={len(y_train)} clips | Val={len(y_val)} clips")


## 3. Kinematic Boost Augmentation, Cosine Classifier & Neural Backbones


In [ ]:
def augment_pose_boosted(seq: np.ndarray, rng: Optional[np.random.Generator] = None) -> np.ndarray:
    # Enhanced kinematic augmentation preserving ASL semantic structure
    gen = rng if rng is not None else np.random.default_rng()
    t_len = seq.shape[0]

    # 1. Temporal elastic resample (0.80 to 1.0)
    ratio = float(gen.uniform(0.80, 1.0))
    win = max(1, min(t_len, int(round(t_len * ratio))))
    st = int(gen.integers(0, t_len - win + 1)) if win < t_len else 0
    idxs = np.round(np.linspace(st, st + win - 1, t_len)).astype(np.int64)
    aug = seq[idxs].astype(np.float32, copy=True)

    # 2. Global coordinate scale jitter (0.88 to 1.12)
    aug[:, :296] *= float(gen.uniform(0.88, 1.12))

    # 3. Spatial translation shift on upper-body / global coords (0..106: 53 keypoints x 2)
    shift = gen.normal(0.0, 0.04, size=(1, 1, 2)).astype(np.float32)
    aug[:, :106].reshape(t_len, 53, 2)[:] += shift

    # 4. Hand-specific scale jitter (106..190: 42 wrist-relative finger coords x 2)
    hand_scale = float(gen.uniform(0.90, 1.10))
    aug[:, 106:190] *= hand_scale

    # 5. Mild Gaussian jitter noise on coordinates
    aug[:, :296] += gen.normal(0.0, 0.015, size=(t_len, 296)).astype(np.float32)

    # 6. Joint confidence dropout
    drop = gen.random((t_len, 53)) < 0.06
    aug[:, 296:][drop] = 0.0

    # 7. Dynamically recompute velocity slice from jittered coordinates
    nc = aug[:, :106].reshape(t_len, 53, 2)
    vl = aug[:, 190:296].reshape(t_len, 53, 2)
    vl[0] = 0.0
    if t_len > 1:
        vl[1:] = np.clip(nc[1:] - nc[:-1], -3.0, 3.0)

    return np.nan_to_num(aug, nan=0.0, posinf=0.0, neginf=0.0)


class PoseDataset(Dataset):
    def __init__(self, p_arr: np.ndarray, y_arr: np.ndarray, augment: bool = False):
        self.p = p_arr
        self.y = y_arr
        self.augment = augment

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        p_seq = np.array(self.p[idx], dtype=np.float32, copy=True)
        if self.augment:
            p_seq = augment_pose_boosted(p_seq)
        return torch.from_numpy(p_seq), torch.tensor(int(self.y[idx]), dtype=torch.long)


class CosineClassifier(nn.Module):
    # Cosine similarity classifier with learnable scale to prevent class norm collapse

    def __init__(self, in_features: int, num_classes: int, init_scale: float = 16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.logit_scale = nn.Parameter(
            torch.tensor(math.log(max(init_scale, 1.0)), dtype=torch.float32)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = F.normalize(x, p=2, dim=-1)
        w_norm = F.normalize(self.weight, p=2, dim=-1)
        scale = self.logit_scale.exp().clamp(min=1.0, max=50.0)
        return scale * F.linear(x_norm, w_norm)


class TemporalAttentionPool(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.proj(seq), dim=1)
        return (seq * weights).sum(dim=1)


class TemporalTransformerEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        embed_dim: int = 192,
        num_layers: int = 2,
        num_heads: int = 4,
        dim_feedforward: int = 384,
        dropout: float = 0.35,
        max_seq_len: int = 128,
    ):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        pe = torch.zeros(1, max_seq_len, embed_dim)
        pos = torch.arange(0, max_seq_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, embed_dim, 2, dtype=torch.float32) * (-math.log(10000.0) / embed_dim)
        )
        pe[0, :, 0::2] = torch.sin(pos * div)
        pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer("pos_encoding", pe, persistent=False)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.attn_pool = TemporalAttentionPool(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        t_len = x.size(1)
        h = self.input_proj(x) + self.pos_encoding[:, :t_len, :]
        return self.attn_pool(self.norm(self.encoder(h)))


class TemporalBiLSTMEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 96,
        num_layers: int = 2,
        dropout: float = 0.35,
    ):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.norm = nn.LayerNorm(hidden_dim * 2)
        self.attn_pool = TemporalAttentionPool(hidden_dim * 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        seq, _ = self.lstm(h)
        return self.attn_pool(self.norm(seq))


class TextProjectionHead(nn.Module):
    def __init__(self, input_dim: int = 384, embed_dim: int = 192, dropout: float = 0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=-1)


class PoseBoostedASLModel(nn.Module):
    def __init__(
        self,
        num_classes: int = 100,
        *,
        pose_dim: int = 349,
        text_dim: int = 384,
        hidden_dim: int = 192,
        embed_dim: int = 192,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.35,
        backbone: str = "transformer",
        mode: str = "pose_only",
        use_cosine_classifier: bool = True,
        fusion_alpha: float = 0.5,
        init_temperature: float = 0.1,
    ):
        super().__init__()
        self.mode = mode
        self.backbone = backbone
        self.use_cosine_classifier = use_cosine_classifier
        self.fusion_alpha = fusion_alpha

        if backbone == "bilstm":
            self.pose_encoder = TemporalBiLSTMEncoder(
                input_dim=pose_dim,
                hidden_dim=hidden_dim // 2,
                num_layers=num_layers,
                dropout=dropout,
            )
        else:
            self.pose_encoder = TemporalTransformerEncoder(
                input_dim=pose_dim,
                embed_dim=hidden_dim,
                num_layers=num_layers,
                num_heads=num_heads,
                dim_feedforward=hidden_dim * 2,
                dropout=dropout,
            )

        self.motion_projection = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

        if use_cosine_classifier:
            self.classifier_head = nn.Sequential(
                nn.Dropout(dropout),
                CosineClassifier(hidden_dim, num_classes),
            )
        else:
            self.classifier_head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, num_classes),
            )

        if mode == "pose_language":
            self.text_projection = TextProjectionHead(text_dim, embed_dim, dropout=0.15)
            self.logit_scale = nn.Parameter(
                torch.tensor(math.log(1.0 / max(init_temperature, 1e-3)), dtype=torch.float32)
            )
            self.lang_fusion_gate = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.Sigmoid(),
            )
            if use_cosine_classifier:
                self.fused_head = nn.Sequential(
                    nn.Dropout(dropout),
                    CosineClassifier(embed_dim, num_classes),
                )
            else:
                self.fused_head = nn.Sequential(
                    nn.Dropout(dropout),
                    nn.Linear(embed_dim, num_classes),
                )
        else:
            self.text_projection = None
            self.register_parameter("logit_scale", None)
            self.lang_fusion_gate = None
            self.fused_head = None

    def forward(
        self,
        pose_inputs: torch.Tensor,
        vocab_text_embeddings: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        repr_vec = self.pose_encoder(pose_inputs)
        if self.mode == "pose_only":
            return {"logits": self.classifier_head(repr_vec), "representation": repr_vec}

        vis_emb = F.normalize(self.motion_projection(repr_vec), p=2, dim=-1)
        txt_emb = self.text_projection(vocab_text_embeddings)
        scale = self.logit_scale.exp().clamp(max=30.0)
        cosine_logits = scale * (vis_emb @ txt_emb.T)
        attn = torch.softmax(cosine_logits, dim=-1)
        lang_ctx = attn @ txt_emb
        gate = self.lang_fusion_gate(torch.cat([vis_emb, lang_ctx], dim=-1))
        fused = gate * vis_emb + (1.0 - gate) * lang_ctx
        cls_logits = self.fused_head(fused) + self.fusion_alpha * cosine_logits

        return {
            "logits": cls_logits,
            "cosine_logits": cosine_logits,
            "visual_embeddings": vis_emb,
            "text_embeddings": txt_emb,
            "logit_scale": scale,
        }


## 4. Train Boosted Models on Validation (100 Epochs, Cosine Schedule)


In [ ]:
def compute_metrics(y_true: np.ndarray, logits: np.ndarray) -> Dict[str, Any]:
    y_t = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(logits, dtype=np.float64)
    y_pred = np.argmax(scores, axis=1)
    top1 = float(np.mean(y_pred == y_t))
    top5 = float(np.mean(np.any(np.argpartition(-scores, kth=4, axis=1)[:, :5] == y_t[:, None], axis=1)))
    f1s = []
    for cid in range(scores.shape[1]):
        tp = int(np.sum((y_t == cid) & (y_pred == cid)))
        fp = int(np.sum((y_t != cid) & (y_pred == cid)))
        fn = int(np.sum((y_t == cid) & (y_pred != cid)))
        if tp + fn == 0:
            continue
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn)
        f1s.append((2.0 * p * r / (p + r)) if (p + r) > 0 else 0.0)
    return {
        "top1_accuracy": top1,
        "top5_accuracy": top5,
        "macro_f1": float(np.mean(f1s)),
        "unique_predicted_classes": int(len(np.unique(y_pred))),
    }


def compute_retrieval(sim: np.ndarray, q_lbl: np.ndarray, g_lbl: np.ndarray) -> Dict[str, float]:
    order = np.argsort(-sim, axis=1)
    ranks = np.asarray(
        [int(np.flatnonzero(g_lbl[o] == q_lbl[i])[0]) + 1 for i, o in enumerate(order)],
        dtype=np.float64,
    )
    return {
        "mrr": float(np.mean(1.0 / ranks)),
        "recall@1": float(np.mean(ranks <= 1)),
        "recall@5": float(np.mean(ranks <= 5)),
    }


def multi_pos_infonce(v: torch.Tensor, t: torch.Tensor, s: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    lg = s * (v @ t.T)
    same = (y.unsqueeze(1) == y.unsqueeze(0)).float()
    tgt = same / same.sum(dim=1, keepdim=True).clamp(min=1.0)
    return 0.5 * (
        -(tgt * F.log_softmax(lg, dim=1)).sum(dim=1).mean()
        - (tgt.T * F.log_softmax(lg.T, dim=1)).sum(dim=1).mean()
    )


def run_boosted_exp(
    exp_id: str,
    backbone: str,
    mode: str,
    *,
    use_cosine_classifier: bool = True,
    hidden_dim: int = 192,
    dropout: float = 0.35,
    lr: float = 8e-4,
    weight_decay: float = 5e-3,
    epochs: int = 100,
    patience: int = 30,
) -> Dict[str, Any]:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    model = PoseBoostedASLModel(
        num_classes=NUM_CLASSES,
        pose_dim=POSE_DIM,
        text_dim=384,
        hidden_dim=hidden_dim,
        embed_dim=hidden_dim,
        dropout=dropout,
        backbone=backbone,
        mode=mode,
        use_cosine_classifier=use_cosine_classifier,
    ).to(DEVICE)

    run_dir = CKPT_DIR / exp_id
    run_dir.mkdir(parents=True, exist_ok=True)

    counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(np.float64)
    w_s = (1.0 / np.sqrt(np.maximum(counts, 1.0)))[y_train]
    sampler = WeightedRandomSampler(weights=w_s.tolist(), num_samples=len(y_train), replacement=True)

    tr_loader = DataLoader(PoseDataset(P_train, y_train, augment=True), batch_size=32, sampler=sampler)
    va_loader = DataLoader(PoseDataset(P_val, y_val, augment=False), batch_size=32, shuffle=False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda ep: (ep + 1) / 5.0 if ep < 5 else 0.5 * (1.0 + math.cos(math.pi * (ep - 5) / max(1, epochs - 5))),
    )

    best_score = (-1.0, float("-inf"))
    best_ep = 0
    best_m: Dict[str, Any] = {}
    best_val_probs: Optional[np.ndarray] = None
    no_imp = 0
    hist = []
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        tr_l, tr_n = 0.0, 0
        for pb, yb in tr_loader:
            pb, yb = pb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            out = model(pb, vocab_text_embeddings=vocab_prompt if mode == "pose_language" else None)
            if mode == "pose_only":
                loss = F.cross_entropy(out["logits"], yb, label_smoothing=0.08)
            else:
                ce = F.cross_entropy(out["logits"], yb, label_smoothing=0.08)
                vc = F.cross_entropy(out["cosine_logits"], yb, label_smoothing=0.08)
                inf = multi_pos_infonce(out["visual_embeddings"], out["text_embeddings"][yb], out["logit_scale"], yb)
                loss = 0.5 * ce + 0.5 * vc + 0.25 * inf

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_l += float(loss.detach()) * yb.size(0)
            tr_n += yb.size(0)

        sched.step()

        model.eval()
        va_l, va_n, lg_all, vis_all, lbl_all, txt_p = 0.0, 0, [], [], [], None
        with torch.inference_mode():
            for pb, yb in va_loader:
                pb, yb = pb.to(DEVICE), yb.to(DEVICE)
                out = model(pb, vocab_text_embeddings=vocab_prompt if mode == "pose_language" else None)
                loss = F.cross_entropy(out["logits"], yb, label_smoothing=0.08)
                va_l += float(loss.detach()) * yb.size(0)
                va_n += yb.size(0)
                lg_all.append(out["logits"].cpu().numpy())
                lbl_all.append(yb.cpu().numpy())
                if mode != "pose_only":
                    vis_all.append(out["visual_embeddings"].cpu().numpy())
                    txt_p = out["text_embeddings"].cpu().numpy()

        lg_np = np.concatenate(lg_all)
        lbl_np = np.concatenate(lbl_all)
        vm = compute_metrics(lbl_np, lg_np)
        vm["loss"] = va_l / max(1, va_n)

        # Softmax probabilities for ensembling
        val_probs_curr = np.exp(lg_np - np.max(lg_np, axis=1, keepdims=True))
        val_probs_curr /= np.sum(val_probs_curr, axis=1, keepdims=True)

        if mode != "pose_only" and txt_p is not None:
            sim = np.concatenate(vis_all) @ txt_p.T
            rv2t = compute_retrieval(sim, lbl_np, np.arange(NUM_CLASSES))
            vm["v2t_recall@1"] = rv2t["recall@1"]
            vm["v2t_recall@5"] = rv2t["recall@5"]
            vm["v2t_mrr"] = rv2t["mrr"]

        hist.append({
            "epoch": ep,
            "train_loss": tr_l / tr_n,
            "val_loss": vm["loss"],
            "val_top1": vm["top1_accuracy"],
            "val_top5": vm["top5_accuracy"],
            "val_macro_f1": vm["macro_f1"],
        })

        sc = (vm["top1_accuracy"], -vm["loss"])
        if sc > best_score:
            best_score, best_ep, best_m, no_imp = sc, ep, dict(vm), 0
            best_val_probs = np.array(val_probs_curr, copy=True)
            ckpt = {
                "epoch": ep,
                "model_state_dict": model.state_dict(),
                "val_metrics": best_m,
                "class_to_id": class_to_id,
                "id_to_class": id_to_class,
                "config": {
                    "pose_dim": POSE_DIM,
                    "hidden_dim": hidden_dim,
                    "embed_dim": hidden_dim,
                    "dropout": dropout,
                    "backbone": backbone,
                    "mode": mode,
                    "use_cosine_classifier": use_cosine_classifier,
                },
            }
            if mode != "pose_only":
                ckpt["vocab_text_embeddings"] = vocab_prompt.detach().cpu()
            torch.save(ckpt, run_dir / "best_model.pt")
            np.save(run_dir / "val_probabilities.npy", best_val_probs)
        else:
            no_imp += 1
            if no_imp >= patience:
                break

    pd.DataFrame(hist).to_csv(run_dir / "training_history.csv", index=False)
    r5 = f" | V2T_R@5={best_m['v2t_recall@5']*100:5.2f}%" if mode != "pose_only" else ""
    print(
        f"[{exp_id:28s}] best_ep={best_ep:2d} | val_loss={best_m['loss']:.3f} | "
        f"val_top1={best_m['top1_accuracy']*100:5.2f}% | val_top5={best_m['top5_accuracy']*100:5.2f}% | "
        f"f1={best_m['macro_f1']:.4f}{r5} | uniq={best_m['unique_predicted_classes']:3d}/100 | {time.time()-t0:.1f}s"
    )

    return {
        "experiment_id": exp_id,
        "backbone": backbone,
        "mode": mode,
        "use_cosine_classifier": use_cosine_classifier,
        "history": hist,
        "val_metrics": best_m,
        "val_probs": best_val_probs,
        "ckpt_path": str(run_dir / "best_model.pt"),
    }


# Train 3 diverse models
r_trans_cos = run_boosted_exp(
    "pose_transformer_cosine",
    backbone="transformer",
    mode="pose_only",
    use_cosine_classifier=True,
    lr=8e-4,
    epochs=100,
    patience=30,
)

r_bilstm_attn = run_boosted_exp(
    "pose_bilstm_attn",
    backbone="bilstm",
    mode="pose_only",
    use_cosine_classifier=True,
    lr=1e-3,
    epochs=100,
    patience=30,
)

r_mm_boost = run_boosted_exp(
    "mm_pose_language_boosted",
    backbone="transformer",
    mode="pose_language",
    use_cosine_classifier=True,
    lr=8e-4,
    epochs=100,
    patience=30,
)


## 5. Multi-Inductive-Bias Inference Ensemble


In [ ]:
import json

# Extract validation probability matrices (N_val, 100)
p1 = r_trans_cos["val_probs"]      # Transformer + Cosine (Pose-Only)
p2 = r_bilstm_attn["val_probs"]     # BiLSTM + Attention + Cosine (Pose-Only)
p3 = r_mm_boost["val_probs"]        # Transformer + MiniLM + Cosine (Multimodal)

# Evaluate combinations on validation
ensemble_candidates = {
    "ensemble_trans_mm": (
        0.50 * p1 + 0.50 * p3,
        {"pose_transformer_cosine": 0.50, "mm_pose_language_boosted": 0.50},
    ),
    "ensemble_trans_bilstm": (
        0.50 * p1 + 0.50 * p2,
        {"pose_transformer_cosine": 0.50, "pose_bilstm_attn": 0.50},
    ),
    "ensemble_all_three": (
        0.40 * p1 + 0.20 * p2 + 0.40 * p3,
        {"pose_transformer_cosine": 0.40, "pose_bilstm_attn": 0.20, "mm_pose_language_boosted": 0.40},
    ),
    "ensemble_equal_three": (
        (p1 + p2 + p3) / 3.0,
        {"pose_transformer_cosine": 1/3, "pose_bilstm_attn": 1/3, "mm_pose_language_boosted": 1/3},
    ),
}

ens_results = {}
for name, (probs, weights) in ensemble_candidates.items():
    m = compute_metrics(y_val, probs)
    ens_results[name] = {"metrics": m, "weights": weights, "probs": probs}
    print(
        f"[{name:28s}] val_top1={m['top1_accuracy']*100:5.2f}% | "
        f"val_top5={m['top5_accuracy']*100:5.2f}% | f1={m['macro_f1']:.4f} | "
        f"uniq={m['unique_predicted_classes']:3d}/100"
    )

# Select best ensemble
best_ens_name = max(ens_results.keys(), key=lambda k: (ens_results[k]["metrics"]["top1_accuracy"], ens_results[k]["metrics"]["top5_accuracy"]))
best_ens = ens_results[best_ens_name]
print(f"\nWinning Validation Ensemble: {best_ens_name} (Top-1={best_ens['metrics']['top1_accuracy']*100:.2f}%, Top-5={best_ens['metrics']['top5_accuracy']*100:.2f}%)")

# Save ensemble configuration for Notebook 05 locked evaluation
ens_dir = CKPT_DIR / "ensemble_boosted"
ens_dir.mkdir(parents=True, exist_ok=True)
with open(ens_dir / "ensemble_config.json", "w") as f:
    json.dump({
        "winning_ensemble_name": best_ens_name,
        "weights": best_ens["weights"],
        "models": {
            "pose_transformer_cosine": r_trans_cos["ckpt_path"],
            "pose_bilstm_attn": r_bilstm_attn["ckpt_path"],
            "mm_pose_language_boosted": r_mm_boost["ckpt_path"],
        },
        "val_metrics": best_ens["metrics"],
    }, f, indent=2)
print(f"Saved ensemble metadata: {ens_dir / 'ensemble_config.json'}")


## 6. Training Curves, Results Tracker & Final Summary


In [ ]:
# 3-Panel Comparison Curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
curve_configs = [
    (r_trans_cos, "Transformer + Cosine", "#1f77b4"),
    (r_bilstm_attn, "BiLSTM + Attn + Cosine", "#2ca02c"),
    (r_mm_boost, "MM Transformer + MiniLM + Cosine", "#ff7f0e"),
]

for res, lbl, col in curve_configs:
    hdf = pd.DataFrame(res["history"])
    axes[0].plot(hdf["epoch"], hdf["val_loss"], label=lbl, color=col)
    axes[1].plot(hdf["epoch"], hdf["val_top1"] * 100.0, label=lbl, color=col)
    axes[2].plot(hdf["epoch"], hdf["val_top5"] * 100.0, label=lbl, color=col)

# Annotate best ensemble on the accuracy panels
axes[1].axhline(best_ens["metrics"]["top1_accuracy"] * 100.0, color="#d62728", linestyle="--", label=f"Best Ens ({best_ens_name})", alpha=0.8)
axes[2].axhline(best_ens["metrics"]["top5_accuracy"] * 100.0, color="#d62728", linestyle="--", label=f"Best Ens ({best_ens_name})", alpha=0.8)

axes[0].set_title("Validation Loss")
axes[1].set_title("Validation Top-1 Accuracy (%)")
axes[2].set_title("Validation Top-5 Accuracy (%)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plot_path = PLOTS_DIR / "04c_pose_boost_curves.png"
plt.savefig(plot_path, dpi=130)
plt.show()

# Update experiments/results_summary.csv
summary_rows = []
for res, mdl_name, mod_name in [
    (r_trans_cos, "PoseTransformerCosine", "pose_only"),
    (r_bilstm_attn, "PoseBiLSTMAttnCosine", "pose_only"),
    (r_mm_boost, "MMPoseLanguageBoosted", "multimodal_pose"),
]:
    vm = res["val_metrics"]
    summary_rows.append({
        "experiment_id": res["experiment_id"],
        "model": mdl_name,
        "modality": mod_name,
        "fusion_mode": res["mode"],
        "split": "val",
        "top1_accuracy": round(vm["top1_accuracy"], 4),
        "top5_accuracy": round(vm["top5_accuracy"], 4),
        "macro_f1": round(vm["macro_f1"], 4),
        "v2t_recall@1": round(vm["v2t_recall@1"], 4) if "v2t_recall@1" in vm else np.nan,
        "v2t_recall@5": round(vm["v2t_recall@5"], 4) if "v2t_recall@5" in vm else np.nan,
        "t2v_recall@1": round(vm["t2v_recall@1"], 4) if "t2v_recall@1" in vm else np.nan,
        "mrr": round(vm["v2t_mrr"], 4) if "v2t_mrr" in vm else np.nan,
        "checkpoint_path": res["ckpt_path"],
    })

# Add best ensemble to summary
summary_rows.append({
    "experiment_id": best_ens_name,
    "model": f"Ensemble({best_ens_name})",
    "modality": "ensemble_kinematic_multimodal",
    "fusion_mode": "probability_blend",
    "split": "val",
    "top1_accuracy": round(best_ens["metrics"]["top1_accuracy"], 4),
    "top5_accuracy": round(best_ens["metrics"]["top5_accuracy"], 4),
    "macro_f1": round(best_ens["metrics"]["macro_f1"], 4),
    "v2t_recall@1": np.nan,
    "v2t_recall@5": np.nan,
    "t2v_recall@1": np.nan,
    "mrr": np.nan,
    "checkpoint_path": str(ens_dir / "ensemble_config.json"),
})

new_df = pd.DataFrame(summary_rows)
res_csv = EXP_DIR / "results_summary.csv"
if res_csv.is_file():
    old_df = pd.read_csv(res_csv)
    old_df = old_df[~((old_df["experiment_id"].isin(new_df["experiment_id"])) & (old_df["split"] == "val"))]
    frames = [df.dropna(axis=1, how="all") for df in (old_df, new_df) if not df.empty]
    merged_df = pd.concat(frames, ignore_index=True)
else:
    merged_df = new_df
merged_df.to_csv(res_csv, index=False)

print("=" * 72)
print("NOTEBOOK 04c (ADVANCED POSE BOOST & ENSEMBLING ON VAL) COMPLETED")
print("=" * 72)
disp_cols = ["experiment_id", "top1_accuracy", "top5_accuracy", "macro_f1"]
if "v2t_recall@5" in new_df:
    disp_cols.append("v2t_recall@5")
if "mrr" in new_df:
    disp_cols.append("mrr")
print(new_df[disp_cols].to_string(index=False))
print(f"\nSaved curves plot   : {plot_path}")
print(f"Updated results CSV : {res_csv}")
print(f"Saved ensemble plan : {ens_dir / 'ensemble_config.json'}")
